In [6]:
import datetime
import polars as pl
from worker.database import connect
import json


DATE_END = datetime.date(2025, 6, 30)
DATE_START = datetime.date(1990, 1, 1)



In [ ]:


with connect() as db:
    df_portfolios = pl.read_database("SELECT * FROM portfolios", db)
    df_investments = pl.read_database(
        """
        SELECT 
            portfolio_id::TEXT as portfolio_id,
            instrument_id::TEXT as instrument_id,
            date,
            quantity
        FROM investments WHERE date <= :date
        """,
        db,
        execute_options={"params": {"date": DATE_END}},
    )

    df_all_dates = pl.DataFrame({"date": s_cal_dates})

    df_market_data = pl.read_database(
        """
            SELECT
                date,
                instrument_id::TEXT as instrument_id,
                value
            FROM market_data
            WHERE date <= :date_max 
            AND date >= :date_min 
            AND data_type = 'adj_close'""",
        db,
        execute_options={"params": {"date_min": DATE_START, "date_max": DATE_END}},
    )

# Build portfolio compositions and values for a range of dates
pairs = df_investments.select("portfolio_id", "instrument_id").unique()

dfs = []
for portfolio_id, instrument_id in pairs.iter_rows():
    df = (
        pl.DataFrame({"portfolio_id": portfolio_id, "instrument_id": instrument_id, "date": s_cal_dates})
        .join(df_investments, on=["portfolio_id", "instrument_id", "date"], how="left")
        .sort("date")
        .with_columns(quantity=pl.col("quantity").replace(None, 0).cum_sum())
        .filter((pl.col("quantity") != 0).cum_sum() > 0)
    )

    dfs.append(df)

df_all_ptf_compositions = pl.concat(dfs)

df_ptf_values = (
    df_all_ptf_compositions.filter(pl.col("date").dt.is_business_day())
    .join(df_market_data, on=["date", "instrument_id"], how="left")
    .group_by("portfolio_id", "instrument_id")
    .map_groups(lambda df: df.sort("date").with_columns(pl.col("value").forward_fill()))
    .filter(pl.col("value").is_not_null())
    .rename({"value": "price"})
    .with_columns(value=pl.col("quantity") * pl.col("price"))
)


# For a single date
df_positions_single_date = (
    df_investments.filter(pl.col("date") <= DATE_END)
    .group_by("portfolio_id", "instrument_id")
    .agg(quantity=pl.col("quantity").sum())
    .with_columns(date=pl.lit(DATE_END))
)

df_values_single_date = (
    df_positions_single_date.join(df_market_data, how="left", on=["instrument_id", "date"])
    .rename({"value": "price"})
    .with_columns(value=pl.col("quantity") * pl.col("price"))
)

u_ptfs = df_values_single_date["portfolio_id"].unique()

positions_json = {
    "date": DATE_END.strftime("%Y-%m-%d"),
    "data": {
        ptf: df_values_single_date.filter(pl.col("portfolio_id") == ptf)
        .drop("portfolio_id", "date", "price")
        .with_columns(pl.selectors.decimal().cast(pl.Float64))
        .to_dicts()
        for ptf in u_ptfs
    },
}

with open("ptfs.json", "w") as fd:
    json.dump(positions_json, fd)


In [15]:
from sqlalchemy.dialects.postgresql import insert
from sqlalchemy.orm import Session

from worker.database import Base, PortfolioComposition, PortfolioValue

def _insert_list_of_dicts(items: list[dict], table: type[Base], session: Session):
    chunk_size = 1000
    total_inserted = 0
    
    for i in range(0, len(items), chunk_size):
        chunk = items[i:i+chunk_size]
        
        try:
            stmt = insert(table).values(chunk)
            stmt = stmt.on_conflict_do_nothing()
            result = session.execute(stmt)
            session.commit()
            
            chunk_inserted = result.rowcount
            total_inserted += chunk_inserted
            
            print(f"Chunk {i//chunk_size + 1}: Inserted {chunk_inserted}/{len(chunk)} rows (Total: {total_inserted})")
            
        except Exception as e:
            print(f"Error inserting chunk {i//chunk_size + 1}: {e}")
            session.rollback()
                        

def fill_portfolio_positions_for_single_date(session, date: datetime.date):
    df_investments = pl.read_database(
        """
        SELECT 
            portfolio_id::TEXT as portfolio_id,
            instrument_id::TEXT as instrument_id,
            date,
            quantity
        FROM investments WHERE date <= :date
        """,
        session,
        execute_options={"params": {"date": DATE_END}},
    )
    
    df_positions_single_date = (
        df_investments.filter(pl.col("date") <= DATE_END)
        .group_by("portfolio_id", "instrument_id")
        .agg(quantity=pl.col("quantity").sum())
        .with_columns(date=pl.lit(DATE_END))
    )

    _insert_list_of_dicts(items=df_positions_single_date.to_dicts(), table=PortfolioComposition, session=session)


    session.commit()

def fill_portfolio_values_multiple_dates(session, date_start: datetime.date, date_end: datetime.date):
    s_cal_dates = pl.date_range(date_start, date_end, eager=True)

    df_investments = pl.read_database(
        """
        SELECT 
            portfolio_id::TEXT as portfolio_id,
            instrument_id::TEXT as instrument_id,
            date,
            quantity
        FROM investments WHERE date <= :date
        """,
        db,
        execute_options={"params": {"date": DATE_END}},
    )

    df_market_data = pl.read_database(
        """
            SELECT
                date,
                instrument_id::TEXT as instrument_id,
                value
            FROM market_data
            WHERE date <= :date_max 
            AND date >= :date_min 
            AND data_type = 'adj_close'""",
        db,
        execute_options={"params": {"date_min": date_start, "date_max": date_end}},
    )

    # Build portfolio compositions and values for a range of dates
    pairs = df_investments.select("portfolio_id", "instrument_id").unique()

    dfs = []
    for portfolio_id, instrument_id in pairs.iter_rows():
        df = (
            pl.DataFrame({"portfolio_id": portfolio_id, "instrument_id": instrument_id, "date": s_cal_dates})
            .join(df_investments, on=["portfolio_id", "instrument_id", "date"], how="left")
            .sort("date")
            .with_columns(quantity=pl.col("quantity").replace(None, 0).cum_sum())
            .filter((pl.col("quantity") != 0).cum_sum() > 0)
        )

        dfs.append(df)

    df_all_ptf_compositions = pl.concat(dfs)

    df_ptf_values = (
        df_all_ptf_compositions.filter(pl.col("date").dt.is_business_day())
        .join(df_market_data, on=["date", "instrument_id"], how="left")
        .group_by("portfolio_id", "instrument_id")
        .map_groups(lambda df: df.sort("date").with_columns(pl.col("value").forward_fill()))
        .filter(pl.col("value").is_not_null())
        .rename({"value": "price"})
        .with_columns(value=pl.col("quantity") * pl.col("price"))
        .group_by("portfolio_id", "date")
        .agg(value=pl.col("value").sum())
    )

    _insert_list_of_dicts(items=df_ptf_values.to_dicts(), table=PortfolioValue, session=session)

    return df_ptf_values

In [21]:
with connect() as session:
    fill_portfolio_positions_for_single_date(session, date=DATE_END)

Chunk 1: Inserted 0/50 rows (Total: 0)


In [151]:
with connect() as session:
    out = fill_portfolio_values_multiple_dates(session, date_start=DATE_START, date_end=DATE_END)

out

Chunk 1: Inserted 0/1000 rows (Total: 0)
Chunk 2: Inserted 0/1000 rows (Total: 0)
Chunk 3: Inserted 0/1000 rows (Total: 0)
Chunk 4: Inserted 0/1000 rows (Total: 0)
Chunk 5: Inserted 0/1000 rows (Total: 0)
Chunk 6: Inserted 0/1000 rows (Total: 0)
Chunk 7: Inserted 0/1000 rows (Total: 0)
Chunk 8: Inserted 0/1000 rows (Total: 0)
Chunk 9: Inserted 0/1000 rows (Total: 0)
Chunk 10: Inserted 0/1000 rows (Total: 0)
Chunk 11: Inserted 0/1000 rows (Total: 0)
Chunk 12: Inserted 0/1000 rows (Total: 0)
Chunk 13: Inserted 0/1000 rows (Total: 0)
Chunk 14: Inserted 0/1000 rows (Total: 0)
Chunk 15: Inserted 0/1000 rows (Total: 0)
Chunk 16: Inserted 0/1000 rows (Total: 0)
Chunk 17: Inserted 0/1000 rows (Total: 0)
Chunk 18: Inserted 0/1000 rows (Total: 0)
Chunk 19: Inserted 0/1000 rows (Total: 0)
Chunk 20: Inserted 0/1000 rows (Total: 0)
Chunk 21: Inserted 0/1000 rows (Total: 0)
Chunk 22: Inserted 0/1000 rows (Total: 0)
Chunk 23: Inserted 0/1000 rows (Total: 0)
Chunk 24: Inserted 0/1000 rows (Total: 0)
C

portfolio_id,date,value
str,date,"decimal[*,20]"
"""b56a59f5-0e5d-4e2c-92cd-61236f…",2016-03-16,45964.37673380000000000000
"""3a844f39-4f69-452c-a4c1-d988e2…",2023-11-13,76290.00000000000000000000
"""67756108-d9ee-44e0-ac13-0e133d…",2024-06-18,81090.00000000000000000000
"""73b9f260-1100-469b-99a2-3a5f9e…",2017-01-05,69774.59589230000000000000
"""d7da851b-20c5-4de0-9c93-68a9f1…",2021-03-18,64490.00000000000000000000
…,…,…
"""8916932a-fe16-49e1-86b7-e99113…",2023-03-28,42000.00000000000000000000
"""f5764652-eca7-48f2-b621-b5fdff…",2018-11-09,41383.38413340000000000000
"""7bb42e63-eb49-4855-8aec-af08df…",2025-01-27,44810.00000000000000000000


In [128]:
positions_json

{'date': datetime.date(2025, 6, 30),
 'data': {'00e01b8c-84f4-4a73-9345-ed2ad74b96b2': [{'instrument_id': '50d64c2b-1484-41db-8095-d18e20c5f436',
    'quantity': Decimal('1000.00000000'),
    'value': Decimal('55680.00000000000000000000')}],
  'f5764652-eca7-48f2-b621-b5fdff1de41f': [{'instrument_id': '29fcfd80-cc50-483f-b2f1-a7099277e15a',
    'quantity': Decimal('1000.00000000'),
    'value': Decimal('61570.00000000000000000000')}],
  '021c0c4c-f691-45d3-ba82-f2b3a650dff8': [{'instrument_id': 'a42d14ef-4680-4d48-ae4e-c6f753cc2ee1',
    'quantity': Decimal('1000.00000000'),
    'value': Decimal('46310.00000000000000000000')}],
  '73b9f260-1100-469b-99a2-3a5f9e64ed92': [{'instrument_id': '56cfb131-8c13-4e90-a0e9-55f93ece20d1',
    'quantity': Decimal('1000.00000000'),
    'value': Decimal('94770.00000000000000000000')}],
  '67756108-d9ee-44e0-ac13-0e133d3e39c0': [{'instrument_id': '07602a17-b596-47d1-adc2-bda6d3cda617',
    'quantity': Decimal('1000.00000000'),
    'value': Decimal('77

In [122]:
df_ptf_values.filter(pl.col("date") == DATE_END)

portfolio_id,instrument_id,date,quantity,price,value
str,str,date,"decimal[*,8]","decimal[*,10]","decimal[*,20]"
"""b56a59f5-0e5d-4e2c-92cd-61236f…","""559e6146-595c-4ea6-a489-0415f4…",2025-06-30,1000.00000000,48.7200000000,48720.00000000000000000000
"""da0955b6-52f2-4c70-bd60-e3a379…","""bc594e1a-e7b2-4b5f-bf6d-e53283…",2025-06-30,1000.00000000,28.8500000000,28850.00000000000000000000
"""23ad6712-e7e8-4db5-9999-bca22e…","""588cf2ec-8e51-4360-8d55-8c807a…",2025-06-30,1000.00000000,99.2000000000,99200.00000000000000000000
"""267b92c2-aa8a-49fe-bf5c-9a2d2c…","""c6d94933-a248-4df5-a56e-b2c0c0…",2025-06-30,1000.00000000,104.4800000000,104480.00000000000000000000
"""fe1ab00b-670a-44e8-bd02-71b5ef…","""49f41541-e632-46f4-bb42-7cbfaf…",2025-06-30,1000.00000000,66.2000000000,66200.00000000000000000000
…,…,…,…,…,…
"""d7da851b-20c5-4de0-9c93-68a9f1…","""02f2fb63-27d1-43bf-953d-7249c4…",2025-06-30,1000.00000000,60.0300000000,60030.00000000000000000000
"""4daf79fe-feb9-4c5a-ba1f-a67b96…","""6c125ed4-4a90-41c4-b389-226eb7…",2025-06-30,1000.00000000,88.2500000000,88250.00000000000000000000
"""48164b2c-2eca-4d1e-81c7-f8c811…","""b2409b2e-2e42-4c6e-8bc5-0b74ae…",2025-06-30,1000.00000000,81.0600000000,81060.00000000000000000000


In [ ]:
df_values_single_date.drop("price", "quantity", "instrument_id").group_by("portfolio_id", "date").agg(
    value=pl.col("value").sum()
)

portfolio_id,date,value
str,date,"decimal[*,20]"
"""da0955b6-52f2-4c70-bd60-e3a379…",2025-06-30,28850.00000000000000000000
"""8916932a-fe16-49e1-86b7-e99113…",2025-06-30,53750.00000000000000000000
"""16cc08b3-c767-464d-8d9c-573d4d…",2025-06-30,620900.00000000000000000000
"""d36a08ce-9974-4a76-a6bb-7cf37f…",2025-06-30,135040.00000000000000000000
"""7bb42e63-eb49-4855-8aec-af08df…",2025-06-30,45440.00000000000000000000
…,…,…
"""2c6a78e0-86dc-4ad1-9cf7-a4a5f6…",2025-06-30,60560.00000000000000000000
"""021c0c4c-f691-45d3-ba82-f2b3a6…",2025-06-30,46310.00000000000000000000
"""b56a59f5-0e5d-4e2c-92cd-61236f…",2025-06-30,48720.00000000000000000000


In [99]:
df

portfolio_id,instrument_id,date,quantity
str,str,date,"decimal[*,8]"
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",1990-01-01,0.00000000
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",1990-01-02,0.00000000
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",1990-01-03,0.00000000
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",1990-01-04,0.00000000
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",1990-01-05,0.00000000
…,…,…,…
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",2025-06-26,1000.00000000
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",2025-06-27,1000.00000000
"""a9ed8979-b384-46f9-a1d8-71fce6…","""22440153-44ce-4830-a6b1-1d7232…",2025-06-28,1000.00000000


In [87]:
len(pairs)

50

In [67]:
df_all_dates

date
date
1990-01-01
1990-01-02
1990-01-03
1990-01-04
1990-01-05
…
2025-06-26
2025-06-27
2025-06-28


In [47]:
df_investment_values

date,instrument_id,value,portfolio_id,quantity
date,str,"decimal[*,10]",str,"decimal[*,8]"
2018-01-30,"""35a6016b-daf4-40a4-84d1-c5492a…",56.7850882744,null,0.00000000
2022-08-11,"""22440153-44ce-4830-a6b1-1d7232…",112.9300000000,null,0.00000000
2018-01-17,"""8da68d05-90e7-47c9-a779-05fce4…",15.9609769894,null,0.00000000
2020-10-23,"""cdf36a61-82d3-4b48-b581-6d7799…",15.6200000000,null,0.00000000
2016-01-29,"""b4dc9f2c-b477-415f-91a8-36ddf6…",85.5330980050,null,0.00000000
…,…,…,…,…
2021-04-16,"""588cf2ec-8e51-4360-8d55-8c807a…",114.5400000000,null,0.00000000
2021-03-16,"""6c125ed4-4a90-41c4-b389-226eb7…",136.3100000000,null,0.00000000
2017-12-06,"""86e48142-757b-4e0d-8f90-f7d70b…",99.9295975801,null,0.00000000


In [29]:
df_investments

instrument_id
str
"""c6d94933-a248-4df5-a56e-b2c0c0…"
"""56cfb131-8c13-4e90-a0e9-55f93e…"
"""f48b1e9f-e2d6-442a-acf6-2ba362…"
"""559e6146-595c-4ea6-a489-0415f4…"
"""7edd1534-ecb5-4a73-a3b0-4bcfe6…"
…
"""cdf36a61-82d3-4b48-b581-6d7799…"
"""3c61196e-5014-4a5d-87b8-9b413c…"
"""3c7f3c30-4848-48ce-b2a5-3fc1b9…"


In [23]:
df_investments

date,portfolio_id,instrument_id,quantity,id
date,object,object,"decimal[*,8]",object
2000-01-01,267b92c2-aa8a-49fe-bf5c-9a2d2cc74924,c6d94933-a248-4df5-a56e-b2c0c08fe450,1000.00000000,997f3c00-b1fa-4537-b247-e0b5b87caacd
2000-01-01,73b9f260-1100-469b-99a2-3a5f9e64ed92,56cfb131-8c13-4e90-a0e9-55f93ece20d1,1000.00000000,656b13a9-cc85-4b75-a533-c3abd943d92b
2000-01-01,4ff1baf0-73ed-4345-9805-eb35a9a7b318,f48b1e9f-e2d6-442a-acf6-2ba36202a71c,1000.00000000,49d56add-5e02-46b8-a664-183bae83b2b5
2000-01-01,b56a59f5-0e5d-4e2c-92cd-61236fb92c8a,559e6146-595c-4ea6-a489-0415f41911c3,1000.00000000,d4eb3878-8a20-4959-a220-702fb9169da1
2000-01-01,7980bbb5-f4e8-43d5-a0a8-2cde41aa8781,7edd1534-ecb5-4a73-a3b0-4bcfe6edc0e1,1000.00000000,3dca983e-6c7c-4eff-bdd3-f4cb50602890
…,…,…,…,…
2000-01-01,ea975ea7-5bc4-4584-ad79-ea24d16d2b21,cdf36a61-82d3-4b48-b581-6d77990bd275,1000.00000000,a4696e38-897d-49e8-ae42-67a566595ad9
2000-01-01,d36a08ce-9974-4a76-a6bb-7cf37fc22dfa,3c61196e-5014-4a5d-87b8-9b413c9de878,1000.00000000,0b8eb06c-f2ba-4dd2-943f-a886fc74ef26
2000-01-01,ce6f327b-247e-4a50-9352-66bc437f6f7b,3c7f3c30-4848-48ce-b2a5-3fc1b9e31b1e,1000.00000000,d85a1d8b-8002-4341-8ed5-ec0a405159b1


In [24]:
df_market_data

instrument_id,date,data_type,value
object,date,str,"decimal[*,10]"
3c7f3c30-4848-48ce-b2a5-3fc1b9e31b1e,2025-06-30,"""adj_close""",37.5100000000
6c125ed4-4a90-41c4-b389-226eb7e93b07,2025-06-30,"""adj_close""",88.2500000000
491f4737-7acc-4f1a-b486-e6cc014c5e76,2025-06-30,"""adj_close""",110.0400000000
7edd1534-ecb5-4a73-a3b0-4bcfe6edc0e1,2025-06-30,"""adj_close""",67.7800000000
27fb9e0c-145d-400a-aa73-1057ff3cb314,2025-06-30,"""adj_close""",72.6800000000
…,…,…,…
a421b6cb-056c-42eb-96ba-c85c8f35e654,2015-09-17,"""adj_close""",44.8565564290
727b0b7c-268d-42c4-bb48-11020f53958a,2015-09-17,"""adj_close""",32.3867121235
8cdd951a-8d3d-40bb-aa43-761fb1226d81,2015-09-17,"""adj_close""",32.4630764370


In [9]:
df_investments

date,portfolio_id,instrument_id,quantity,id
date,object,object,"decimal[*,8]",object
2000-01-01,267b92c2-aa8a-49fe-bf5c-9a2d2cc74924,c6d94933-a248-4df5-a56e-b2c0c08fe450,1000.00000000,997f3c00-b1fa-4537-b247-e0b5b87caacd
2000-01-01,73b9f260-1100-469b-99a2-3a5f9e64ed92,56cfb131-8c13-4e90-a0e9-55f93ece20d1,1000.00000000,656b13a9-cc85-4b75-a533-c3abd943d92b
2000-01-01,4ff1baf0-73ed-4345-9805-eb35a9a7b318,f48b1e9f-e2d6-442a-acf6-2ba36202a71c,1000.00000000,49d56add-5e02-46b8-a664-183bae83b2b5
2000-01-01,b56a59f5-0e5d-4e2c-92cd-61236fb92c8a,559e6146-595c-4ea6-a489-0415f41911c3,1000.00000000,d4eb3878-8a20-4959-a220-702fb9169da1
2000-01-01,7980bbb5-f4e8-43d5-a0a8-2cde41aa8781,7edd1534-ecb5-4a73-a3b0-4bcfe6edc0e1,1000.00000000,3dca983e-6c7c-4eff-bdd3-f4cb50602890
…,…,…,…,…
2000-01-01,ea975ea7-5bc4-4584-ad79-ea24d16d2b21,cdf36a61-82d3-4b48-b581-6d77990bd275,1000.00000000,a4696e38-897d-49e8-ae42-67a566595ad9
2000-01-01,d36a08ce-9974-4a76-a6bb-7cf37fc22dfa,3c61196e-5014-4a5d-87b8-9b413c9de878,1000.00000000,0b8eb06c-f2ba-4dd2-943f-a886fc74ef26
2000-01-01,ce6f327b-247e-4a50-9352-66bc437f6f7b,3c7f3c30-4848-48ce-b2a5-3fc1b9e31b1e,1000.00000000,d85a1d8b-8002-4341-8ed5-ec0a405159b1
